In [0]:
%run "/Workspace/Users/collinlee-@live.ca/fake_general_ledger/1.1 bronze_ingestion_framework"

In [0]:
%run "/Workspace/Users/collinlee-@live.ca/fake_general_ledger/1.01 bronze_ingestion_audit"

In [0]:
%run "/Workspace/Users/collinlee-@live.ca/fake_general_ledger/1.03 bronze_ingestion_schema_drift_records"

In [0]:
%run "/Workspace/Users/collinlee-@live.ca/fake_general_ledger/1.04 bronze_ingestion_observability_records"

In [0]:
# ---------------------------------------------------------
# 0. Bronze Ingestion Framework Parameters
# ---------------------------------------------------------

control_table = "fake_general_ledger.default.bronze_ingestion_control"

source_system = "gl"

bronze_table = "fake_general_ledger.default.bronze_general_ledger"

audit_table = "fake_general_ledger.default.bronze_ingestion_audit"

schema_log_table = "fake_general_ledger.default.bronze_ingestion_schema_drift"

observability_table = "fake_general_ledger.default.bronze_ingestion_observability"

volume_path = '/Volumes/fake_general_ledger/default/gl_raw'

volume_identifier = 'general_ledger'

pipeline_run_id = str(uuid.uuid4())

batch_id = str(uuid.uuid4())

spark = SparkSession.builder.appName('bronze_general_ledger').getOrCreate()

logger = setup_logging()

In [0]:
# ---------------------------------------------------------
# 1. Bronze Ingestion Driver Code
# ---------------------------------------------------------

files = get_correct_files(
    logger,
    volume_path,
    volume_identifier
)

for file_path in files:

    ingestion_start_timestamp = datetime.now()

    file_hash = generate_file_hash(file_path)

    file_size = get_file_size(file_path)

    if file_already_processed(
        spark,
        control_table,
        source_system,
        bronze_table,
        file_hash
    ):
        continue

    try:

        register_ingestion_start(
            spark,
            control_table,
            source_system,
            bronze_table,
            file_path,
            pipeline_run_id,
            batch_id
        )

        # ---------------------------------------
        # READ SOURCE FILE
        # ---------------------------------------

        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .option("mode", "PERMISSIVE")
            .csv(file_path)
        )

        total_input_rows = df.count()

        # ---------------------------------------
        # EXTRACT INCOMING SCHEMA
        # ---------------------------------------

        incoming_schema = extract_schema_dict(df)

        # ---------------------------------------
        # GET EXISTING TABLE SCHEMA
        # ---------------------------------------

        existing_schema = get_existing_table_schema(
            spark,
            bronze_table
        )

        # ---------------------------------------
        # COMPARE SCHEMAS
        # ---------------------------------------

        schema_changes = compare_schemas(
            incoming_schema,
            existing_schema
        )

        schema_change_count = len(schema_changes)

        # ---------------------------------------
        # WRITE SCHEMA DRIFT LOGS
        # ---------------------------------------

        write_schema_changes(
            spark,
            schema_changes,
            schema_log_table,
            source_system,
            bronze_table,
            batch_id,
            pipeline_run_id,
            file_path
        )

        # ---------------------------------------
        # APPLY BRONZE METADATA
        # ---------------------------------------

        valid_df = apply_bronze_metadata(
            df,
            file_path,
            batch_id,
            file_hash
        )

        valid_rows = valid_df.count()

        # ---------------------------------------
        # WRITE VALID DATA
        # ---------------------------------------

        append_to_bronze_table(
            logger,
            valid_df,
            bronze_table
        )

        register_ingestion_success(
            spark,
            control_table,
            source_system,
            bronze_table,
            file_path,
            valid_df.count()
        )

        ingestion_end_timestamp = datetime.now()

        write_ingestion_observability(
            spark,
            observability_table,
            source_system,
            bronze_table,
            batch_id,
            pipeline_run_id,
            file_path,
            file_size,
            file_hash,
            ingestion_start_timestamp,
            ingestion_end_timestamp,
            total_input_rows,
            valid_rows,
            schema_change_count,
            "SUCCESS"
        )

    except Exception as e:

        ingestion_end_timestamp = datetime.now()

        register_ingestion_failure(
            spark,
            control_table,
            source_system,
            bronze_table,
            file_path,
            str(e)
        )

        write_ingestion_observability(
            spark,
            observability_table,
            source_system,
            bronze_table,
            batch_id,
            pipeline_run_id,
            file_path,
            file_size,
            file_hash,
            ingestion_start_timestamp,
            ingestion_end_timestamp,
            0,
            0,
            0,
            "FAILED",
            str(e)
        )

        raise